# Problem logarytmu dyskretnego

Niech $(G,\circ)$ będzie grupą z działaniem $\circ$ i elementem neutralnym $1_G$. Wtedy dla dowolnego elementu $a\in G$ i $k\in\mathbb{Z}$ definiujemy *potęgę* $$a^k =\left\{\begin{array}{cc}
\underbrace{a\circ a\circ \ldots \circ a}_{k}&\text{ dla }k>0\\
1_G&\text{ dla }k=0\\
\underbrace{a^{-1}\circ a^{-1}\circ \ldots \circ a^{-1}}_{k}&\text{ dla }k<0
\end{array}\right.$$
gdzie $a^{-1}$ jest elementem odwrotnym do $a$.

Dla $a,b\in G$, $b\neq 1_G$, **logarytmem dyskretnym** $\log_b a$ jest każda liczba $k\in\mathbb{Z}$ taka, że $b^k=a$.

## Logarytm dyskretny w $\mathbb{Z}_n$

W przypadku pierścienia $\mathbb{Z}_n$ logarytmem dyskretnym $\log_b a$ jest każda liczba $k\in\mathbb{Z}$ taka, że $b^k=a\bmod n$, o ile w ogóle istnieje.

Specyficzną sytuacją w teorii liczb jest gdy $n=p$ jest liczbą pierwszą a $q$ jest **pierwiastkiem pierwotnym** $\mod p$. Wtedy:
- potęgi $q^k\mod p$ generują cały zbiór $[1,p-1]$, tzn. $q$ jest tzw. **generatorem grupy multiplikatywnej** rzędu $p-1$
- logarytm dyskretny $\log_q a$ istnieje dla każdego niezerowego elementu $a\in \mathbb{Z}_p$

### Skąd wziąć generator grupy multiplikatywnej?

,,Pierwiastek pierwotny (nazywany także pierwiastkiem prymitywnym) to element grupy multiplikatywnej modulo n, którego rząd jest równy rzędowi grupy. W teorii grup oznacza to, że jest to generator grupy – element, który poprzez kolejne potęgowanie generuje wszystkie elementy danej grupy (za Wikipedia).''

W ogólnym przypadku i dla dużych $p$ znalezienie generatora grupy jest obliczeniowo trudne. Najbardziej naiwne postępowanie polega na wylosowaniu liczby $g$ i sprawdzeniu czy podnosząc ją do potęgi ($k\in \mathbb{Z}$) uzyskamy wszystkie elementy grupy. 

Mniej naiwne postępowanie opiera się na własności rzędu elementu w grupie:

1. Rozkład $p-1$ na czynniki pierwsze (aby znaleźć liczby względnie pierwsze z $p$):
$$ p-1 = \prod q_i^{e_i} $$

2. Sprawdzamy liczby z zakresu $g \in \{2,..., p-1\}$ czy spełniają kongruencję:
$g^\frac{p-1}{q_i} \equiv 1 \bmod p$ dla wszystkich czynników $q_i$
    
    a. jeśli przynajmniej raz kongruencja jest spełniona $g$ nie jest generatorem!
    
    b. w przeciwnym wypadku $g$ jest generatorem

## Zadanie 1
Napisz kod znajdujący generatory grupy multiplikatywnej modulo liczba pierwsza $p$ stosując opisaną powyżej metodę. Możesz zastosować metodę odwrotną: wylosuj liczby pierwsze, które będą czynnikami pierwszymi $p-1$, upewnij się jednak czy $p$ jest liczbą pierwszą (!).  

In [1]:
import secrets
import math

def prime_factors(n):
    factors = set()
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.add(d)
            n //= d
        d += 1
    if n > 1:
        factors.add(n)
    return factors

def find_generator(p):
    if p < 2 or not isinstance(p, int):
        raise ValueError("p must be a prime integer >= 2")
    phi = p - 1
    factors = prime_factors(phi)
    for g in range(2, p):
        for q in factors:
            if pow(g, phi // q, p) == 1:
                break
        else:
            return g
    raise ValueError("No generator found")

p = 23 # this can be changed to whatever we want
g = find_generator(p)
print("Generator grupy modulo p:", g)
elements = sorted({pow(g, k, p) for k in range(1, p)})
print("Grupa multiplikatywna mod p:", elements)

Generator grupy modulo p: 5
Grupa multiplikatywna mod p: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


# Wymiana klucza typu Diffie-Hellman

Alice i Bob uzgadniają klucz publiczny będący liczbą pierwszą $p$ oraz $g$ - pierwiastkiem pierwotnym mod $p$.
- sekret Alice: liczba całkowita $n\in \mathbb{Z}_p\setminus\{0\}$
- sekret Boba: liczba całkowita $m\in \mathbb{Z}_p\setminus\{0\}$
- Alice generuje $x=g^n\mod p$ i wysyła do Boba
- Bob generuje $y=g^m\mod p$ i wysyła Alice
- Alice oblicza klucz $k=y^n\mod p$
- Bob oblicza klucz $k=x^m\mod p$


## Zadanie 2.

Zaimplementuj powyższy algorytm wymiany klucza. Użyj parametrów, które otrzymałeś w Zadaniu 1

In [2]:
alice_secret = secrets.randbelow(p - 1) + 1
bob_secret = secrets.randbelow(p - 1) + 1
A = pow(g, alice_secret, p)  # Alice -> Bob
B = pow(g, bob_secret, p)    # Bob -> Alice
alice_key = pow(B, alice_secret, p)
bob_key = pow(A, bob_secret, p)
print("Alice secret:", alice_secret, "Bob secret:", bob_secret)
print("Alice public A:", A, "Bob public B:", B)
print("Shared keys equal:", alice_key == bob_key, "key:", alice_key)


Alice secret: 12 Bob secret: 16
Alice public A: 18 Bob public B: 3
Shared keys equal: True key: 3


## Algorytm baby-step giant-step

Jeden z najprostszych (poza metodą naiwną) algorytmów poszukiwania logarytmu dyskretnego w grupach cyklicznych.

Niech $p$ będzie liczbą pierwszą oraz niech $q$ będzie pierwiastkiem pierwotnym modulo $p$. Dla niezerowego $a\in\mathbb{Z}_p$ szukamy liczby $k\in\mathbb{Z}$ takiej, że $q^k=a\mod p$

### Krok 1.
- $m=\lceil\sqrt{p-1}\rceil$
- tworzymy pomocniczą tablicę potęg: dla wszystkich $i\in [0,m)$ obliczamy parę $(i,q^i)$
- obliczamy $r=(q^{-1})^m$
### Krok 2.
- $b=a$
- dla wszystkich $j\in [0,m)$:
    - sprawdzamy, czy para $(i,b)$ jest elementem tablicy potęg dla pewnego $i$
    - jeżeli tak, to $k=jm+i$ i kończymy algorytm
    - jeżeli nie, to $b=br$ i kontynuujemy pętlę

## Zadanie 3.

Zaimplementować algorytm baby-step giant-step. Przetestować dla podanych danych testujących.

```Dane testujące:
p = 7
q = 3
a = 4

m = 3
tablica_testowa = [1,3,2]
r = 6
k = 4 (j = 1, i = 1)
```

```
p = 29
q = 8
a = 10

m = 6
tablica_testowa = [1,8,6,19,7,27]
r = 9
k = 17 (j = 2, i = 5)
```

```
p = 113
q = 76
a = 84

m = 11
tablica_testowa = [1,76,13,84,56,75,50,71,85,19,88]
r = 70
k = 3 (j = 0, i = 3)
```

In [3]:
def baby_step_giant_step(q, a, p):
    """
    Solve q^k = a (mod p) for k in [0, p-2] using BSGS.
    Returns k or None.
    """
    m = math.isqrt(p - 1) + 1
    # baby steps: q^i -> i
    table = {}
    cur = 1
    for i in range(m):
        if cur not in table:
            table[cur] = i
        cur = (cur * q) % p
    # compute r = (q^{-1})^m
    q_inv = pow(q, -1, p)
    r = pow(q_inv, m, p)
    gamma = a % p
    for j in range(m):
        if gamma in table:
            return j * m + table[gamma]
        gamma = (gamma * r) % p
    return None

In [4]:
tests = [
    (7, 3, 4, 4),
    (29, 8, 10, 17),
    (113, 76, 84, 3),
]
for p_t, q_t, a_t, expected in tests:
    k = baby_step_giant_step(q_t, a_t, p_t)
    print(f"p={p_t}, q={q_t}, a={a_t} -> k={k}, expected={expected}, ok={k==expected}")

p=7, q=3, a=4 -> k=4, expected=4, ok=True
p=29, q=8, a=10 -> k=17, expected=17, ok=True
p=113, q=76, a=84 -> k=3, expected=3, ok=True
